# CS604 latency with zoom — Van

Runtime: **A100 40GB GPU** (the GPU every reported latency uses), High-RAM off. No training and no dataset download: it only reads two checkpoints from **CS604_RUNS** and times them on synthetic inputs (`scripts/van_latency_zoom.py`). About 15 minutes in total.

For each run it writes `inference_latency_zoom.json` into that run's folder: one forward pass and the `--zoom` submission path (two passes + box), batch 1 and 64, bf16, 30 warm-ups + 200 timed iterations. The Task 2 run is also timed with its LoRA merged into the weights (the AdaptFormer branch cannot be merged).

The GitHub repo is private, so the clone cell asks for a Personal Access Token (hidden). Do not paste it into the notebook text.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "van-dev"
# (run folder in CS604_RUNS, extra flags)
RUNS = [
    ("full_ft_20261017_122237", ""),                      # Task 1 final, full FT 448
    ("van_adaptformer_20261021_083823", "--merge_lora"),  # Task 2 final, AdaptFormer + LoRA 448
]
# End the runtime after the last cell so it stops using compute units.
AUTO_DISCONNECT = True


def end_runtime(reason):
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()
        print("Drive flushed and unmounted.")
    finally:
        runtime.unassign()


## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt


## 3. Mount Drive and check the GPU and run folders

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", gpu)
assert "A100" in gpu, "Switch the runtime to an A100 so the numbers match the rest of the report."
for run, _ in RUNS:
    folder = Path(DRIVE_OUT) / run
    assert (folder / "config.yaml").is_file() and (folder / "last.pt").is_file(), f"missing files in {folder}"
    print("found", folder)


## 4. Time each run (a few minutes each)

In [ ]:
import json
import subprocess

failed = []
for run, flags in RUNS:
    cmd = f"python scripts/van_latency_zoom.py --run_dir {DRIVE_OUT}/{run} {flags}"
    print("$", cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        failed.append(run)

for run, _ in RUNS:
    path = Path(DRIVE_OUT) / run / "inference_latency_zoom.json"
    if path.is_file():
        res = json.loads(path.read_text())
        print()
        print(f"{run} ({res['gpu']}, {res['img_size']} px)")
        for key in ("single", "zoom", "merged_single", "merged_zoom"):
            if key in res:
                print(f"  {key:14s} " + " | ".join(f"bs{r['batch_size']} {r['ms_per_image']:.2f} ms/image" for r in res[key]))
print("failed:", failed or "none")


## 5. End the runtime

In [ ]:
end_runtime("Latency done; ending the runtime.")
